# Stage 3 Inference

Load either an instruction-tuned checkpoint or a base protein language-model checkpoint, then give it an input prompt and generate protein output.

Defaults are conservative for a 16GB GPU. Start with `max_new_tokens=128`, then increase after the first successful run.

In [9]:
import gc
import json
import math
import os
import random
import sys
from pathlib import Path
from typing import Any, Mapping

import torch
import torch.nn.functional as F


def find_repo_dir_for_import(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    repo_dir_env = os.environ.get("MDNAC_REPO_DIR")
    if repo_dir_env:
        candidates.append(Path(repo_dir_env).expanduser())
    candidates.extend([
        Path("/content/MDNAC"),
        Path("/content/drive/MyDrive/MDNAC"),
    ])
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if (resolved / "pyproject.toml").exists() and (resolved / "libs").is_dir():
            return resolved
    raise RuntimeError("Could not locate repo. Run from the repo or set MDNAC_REPO_DIR.")


REPO_DIR = find_repo_dir_for_import(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from libs.core.app import MicrobialDecoderCoreApp
from libs.core.fusion import FusedVocabularyLayout
from libs.core.mdc import MDCDecoderModel
from libs.core.mdc.config import MDCModelConfig
from libs.core.pretrain.distributed import normalize_parallel_state_dict
from libs.core.pretrain.profiled import MDCProfileSequencePretrainArtifacts
from libs.core.pretrain.protein_lm.core import PROTEIN_END_TOKEN, PROTEIN_START_TOKEN
from libs.core.pretrain.protein_lm.support.backbone import is_supported_protein_checkpoint_family
from libs.data.training.tokenizer import SequenceTokenizer
from libs.instruction.schema import format_instruction_prompt
from libs.instruction.trainer import INSTRUCTION_CHECKPOINT_FAMILY
import libs.core.mdc.linear_attention as linear_attention

print(f"Repo: {REPO_DIR}")
print(f"Torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Repo: C:\Users\Admin\Desktop\MDNAC
Torch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 5060 Ti


## Settings

Use `MODEL_KIND = "auto"` first. If an instruction checkpoint exists it will load that; otherwise it will fall back to the best base protein checkpoint.

In [10]:
MODEL_KIND = "instruction"  # "auto", "instruction", or "protein"

INSTRUCTION_CHECKPOINT_CANDIDATES = [
    REPO_DIR / "data/checkpoints/protein_instruction/checkpoint_best.pt",
    REPO_DIR / "data/checkpoints/protein_instruction/checkpoint_last.pt",
    REPO_DIR / "data/checkpoints/protein_instruction/checkpoint_final.pt",
]
PROTEIN_CHECKPOINT_CANDIDATES = [
    REPO_DIR / "data/checkpoints/protein_from_scratch/checkpoint_best.pt",
    REPO_DIR / "data/checkpoints/protein/checkpoint_best.pt",
]
ARTIFACT_DIR = REPO_DIR / "data/compiled/refseq_bacteria_instruction_profile"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MIXED_PRECISION = "auto"  # "auto", "bf16", "fp16", or "no"
USE_FP32_LINEAR_ATTENTION_FALLBACK = False

# Fast first-run defaults for a 16GB GPU. Increase after the first successful output.
DEFAULT_MAX_NEW_TOKENS = 64
DEFAULT_TEMPERATURE = 0.8
DEFAULT_TOP_K = 50
DEFAULT_NUM_CANDIDATES = 1
DEFAULT_PROGRESS_EVERY = 16

linear_attention.use_fp32_fallback_linear_attention = USE_FP32_LINEAR_ATTENTION_FALLBACK


## Loader Helpers

In [11]:
def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.is_file():
            return path
    return None


def resolve_checkpoint_path(model_kind: str) -> tuple[str, Path]:
    normalized = model_kind.lower().strip()
    instruction_path = first_existing(INSTRUCTION_CHECKPOINT_CANDIDATES)
    protein_path = first_existing(PROTEIN_CHECKPOINT_CANDIDATES)

    if normalized == "instruction":
        if instruction_path is None:
            raise FileNotFoundError("No instruction checkpoint found in INSTRUCTION_CHECKPOINT_CANDIDATES.")
        return "instruction", instruction_path
    if normalized == "protein":
        if protein_path is None:
            raise FileNotFoundError("No protein checkpoint found in PROTEIN_CHECKPOINT_CANDIDATES.")
        return "protein", protein_path
    if normalized != "auto":
        raise ValueError("MODEL_KIND must be 'auto', 'instruction', or 'protein'.")
    if instruction_path is not None:
        return "instruction", instruction_path
    if protein_path is not None:
        return "protein", protein_path
    raise FileNotFoundError("No checkpoint found. Update the candidate paths above.")


def coerce_torch_dtype(value: Any) -> torch.dtype:
    if isinstance(value, torch.dtype):
        return value
    text = str(value).replace("torch.", "")
    mapping = {
        "float32": torch.float32,
        "float": torch.float32,
        "float16": torch.float16,
        "half": torch.float16,
        "bfloat16": torch.bfloat16,
    }
    return mapping.get(text, torch.float32)


def model_config_from_checkpoint(checkpoint: Mapping[str, Any]) -> MDCModelConfig:
    payload = dict(checkpoint.get("model_config") or {})
    if not payload:
        raise ValueError("Checkpoint is missing model_config.")
    if payload.get("layer_types") is not None:
        payload["layer_types"] = tuple(payload["layer_types"])
    payload["dtype"] = coerce_torch_dtype(payload.get("dtype", torch.float32))
    return MDCModelConfig(**payload)


def autocast_dtype() -> torch.dtype | None:
    if DEVICE.type != "cuda" or MIXED_PRECISION == "no":
        return None
    if MIXED_PRECISION == "bf16":
        return torch.bfloat16
    if MIXED_PRECISION == "fp16":
        return torch.float16
    if MIXED_PRECISION != "auto":
        raise ValueError("MIXED_PRECISION must be 'auto', 'bf16', 'fp16', or 'no'.")
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def normalize_app_state_dict(state_dict: Mapping[str, Any]) -> dict[str, Any]:
    normalized = dict(state_dict)
    for prefix in ("module.", "_orig_mod."):
        while normalized and all(key.startswith(prefix) for key in normalized):
            normalized = {key[len(prefix):]: value for key, value in normalized.items()}
    return normalized

In [12]:
def load_instruction_runtime(checkpoint_path: Path, artifact_dir: Path | None = None):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if checkpoint.get("model_family") != INSTRUCTION_CHECKPOINT_FAMILY:
        raise ValueError(f"Not an instruction checkpoint: {checkpoint.get('model_family')!r}")

    cfg = model_config_from_checkpoint(checkpoint)
    layout = FusedVocabularyLayout.from_raw_tensor_payload({"config": checkpoint["layout"]})
    app = MicrobialDecoderCoreApp(cfg, layout)
    app.load_state_dict(normalize_app_state_dict(checkpoint["model_state_dict"]), strict=True)
    app.to(DEVICE).eval()

    artifact_dir_candidates = []
    if artifact_dir is not None:
        artifact_dir_candidates.append(Path(artifact_dir))
    artifact_dir_candidates.append(ARTIFACT_DIR)
    checkpoint_artifact_dir = checkpoint.get("artifact_dir")
    if checkpoint_artifact_dir:
        artifact_dir_candidates.append(Path(checkpoint_artifact_dir))

    tried_tokenizer_maps = []
    tokenizer_map_path = None
    for candidate_dir in artifact_dir_candidates:
        candidate_path = candidate_dir / "tokenizer_map.json"
        tried_tokenizer_maps.append(candidate_path)
        if candidate_path.is_file():
            tokenizer_map_path = candidate_path
            break
    if tokenizer_map_path is None:
        tried = "\n  - ".join(str(path) for path in tried_tokenizer_maps)
        raise FileNotFoundError(
            "Missing instruction artifact tokenizer map. Tried:\n"
            f"  - {tried}\n"
            "Run the instruction training notebook first, or set ARTIFACT_DIR."
        )
    artifacts = MDCProfileSequencePretrainArtifacts.from_tokenizer_map_file(tokenizer_map_path)
    resolved_artifact_dir = tokenizer_map_path.parent
    return {
        "kind": "instruction",
        "checkpoint_path": checkpoint_path,
        "checkpoint": checkpoint,
        "model": app,
        "artifacts": artifacts,
        "layout": layout,
        "context_length": int(cfg.context_length),
    }


def load_protein_runtime(checkpoint_path: Path):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if not is_supported_protein_checkpoint_family(checkpoint.get("model_family")):
        raise ValueError(f"Not a supported protein checkpoint: {checkpoint.get('model_family')!r}")

    cfg = model_config_from_checkpoint(checkpoint)
    model = MDCDecoderModel(cfg)
    model.load_state_dict(normalize_parallel_state_dict(checkpoint["model_state_dict"]), strict=True)
    model.to(DEVICE).eval()

    tokenizer = SequenceTokenizer()
    tokenizer._load_from_payload(checkpoint["tokenizer_map"])
    return {
        "kind": "protein",
        "checkpoint_path": checkpoint_path,
        "checkpoint": checkpoint,
        "model": model,
        "tokenizer": tokenizer,
        "context_length": int(cfg.context_length),
    }


kind, checkpoint_path = resolve_checkpoint_path(MODEL_KIND)
print(f"Loading {kind} checkpoint: {checkpoint_path}")
RUNTIME = load_instruction_runtime(checkpoint_path) if kind == "instruction" else load_protein_runtime(checkpoint_path)
print(json.dumps({
    "kind": RUNTIME["kind"],
    "checkpoint_path": str(RUNTIME["checkpoint_path"]),
    "context_length": RUNTIME["context_length"],
    "device": str(DEVICE),
    "autocast_dtype": str(autocast_dtype()),
}, indent=2))

Loading instruction checkpoint: C:\Users\Admin\Desktop\MDNAC\data\checkpoints\protein_instruction\checkpoint_best.pt
{
  "kind": "instruction",
  "checkpoint_path": "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\checkpoints\\protein_instruction\\checkpoint_best.pt",
  "context_length": 512,
  "device": "cuda",
  "autocast_dtype": "torch.bfloat16"
}


## Sampling Helpers

In [13]:
def sample_next_token(logits: torch.Tensor, *, temperature: float, top_k: int | None) -> torch.Tensor:
    if top_k is not None and top_k > 0:
        top_k = min(int(top_k), logits.size(-1))
        top_values, _ = torch.topk(logits, k=top_k, dim=-1)
        threshold = top_values[:, -1].unsqueeze(-1)
        logits = torch.where(logits < threshold, torch.full_like(logits, float("-inf")), logits)

    if temperature and temperature > 0:
        probs = torch.softmax(logits / float(temperature), dim=-1)
        return torch.multinomial(probs, num_samples=1)
    return torch.argmax(logits, dim=-1, keepdim=True)


def set_seed(seed: int | None) -> None:
    if seed is None:
        return
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## Instruction Inference

Use this after `data/checkpoints/protein_instruction/checkpoint_best.pt` exists. The model receives an instruction plus optional input text and predicts a protein sequence.

In [14]:
def fused_instruction_prompt(artifacts, instruction: str, input_text: str = "") -> torch.Tensor:
    layout = artifacts.layout
    profile = format_instruction_prompt(instruction, input_text)
    profile_ids = artifacts.profile_tokenizer.encode(profile, add_bos=True, add_eos=True)

    # Match training fusion: global BOS, profile tokens without source BOS/EOS, separator, then sequence tokens.
    if profile_ids and profile_ids[0] == 1:
        profile_ids = profile_ids[1:]
    if profile_ids and profile_ids[-1] == 2:
        profile_ids = profile_ids[:-1]

    fused_ids = [layout.bos_token_id]
    fused_ids.extend(layout.map_profile_token_id(token_id) for token_id in profile_ids)
    fused_ids.append(layout.sep_token_id)
    return torch.tensor(fused_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)


def mask_to_sequence_vocab(logits: torch.Tensor, layout) -> torch.Tensor:
    masked = torch.full_like(logits, float("-inf"))
    start = int(layout.sequence_offset)
    end = start + int(layout.sequence_vocab_size)
    masked[:, start:end] = logits[:, start:end]
    masked[:, int(layout.eos_token_id)] = logits[:, int(layout.eos_token_id)]
    return masked


def decode_generated_sequence(artifacts, generated_fused_ids: list[int]) -> str:
    layout = artifacts.layout
    sequence_ids = []
    for token_id in generated_fused_ids:
        token_id = int(token_id)
        if token_id == layout.eos_token_id:
            break
        if layout.sequence_offset <= token_id < layout.sequence_offset + layout.sequence_vocab_size:
            sequence_ids.append(token_id - layout.sequence_offset)
    return artifacts.decode_sequence(sequence_ids, skip_special=True)


def generate_instruction_response(
    instruction: str,
    input_text: str = "",
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    temperature: float = DEFAULT_TEMPERATURE,
    top_k: int | None = DEFAULT_TOP_K,
    restrict_to_sequence_vocab: bool = True,
    seed: int | None = None,
    use_cache: bool = True,
    verbose: bool = False,
    progress_every: int = DEFAULT_PROGRESS_EVERY,
) -> dict[str, Any]:
    if RUNTIME["kind"] != "instruction":
        raise RuntimeError("Current RUNTIME is not an instruction checkpoint. Set MODEL_KIND='instruction' after training one.")

    set_seed(seed)
    app = RUNTIME["model"]
    artifacts = RUNTIME["artifacts"]
    layout = artifacts.layout
    prompt_ids = fused_instruction_prompt(artifacts, instruction, input_text)
    prompt_length = int(prompt_ids.size(1))
    max_context = int(RUNTIME["context_length"])
    generated_ids: list[int] = []
    amp_dtype = autocast_dtype()

    if prompt_length >= max_context:
        raise ValueError(f"Prompt length {prompt_length} is already >= context length {max_context}.")

    with torch.no_grad():
        if use_cache:
            cache = app.model.create_kv_cache()
            app.model.reset_kv_cache(cache)
            with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=(amp_dtype is not None)):
                logits = app.model(prompt_ids, cache=cache)[:, -1, :]

            for step in range(int(max_new_tokens)):
                if prompt_length + len(generated_ids) >= max_context:
                    break
                next_logits = mask_to_sequence_vocab(logits, layout) if restrict_to_sequence_vocab else logits
                next_token = sample_next_token(next_logits, temperature=temperature, top_k=top_k)
                next_id = int(next_token.item())
                generated_ids.append(next_id)
                if verbose and progress_every > 0 and (step + 1) % int(progress_every) == 0:
                    print(f"generated {step + 1}/{max_new_tokens} tokens", flush=True)
                if next_id == layout.eos_token_id:
                    break
                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=(amp_dtype is not None)):
                    logits = app.model(next_token, cache=cache)[:, -1, :]
        else:
            token_ids = prompt_ids
            for step in range(int(max_new_tokens)):
                if token_ids.size(1) >= max_context:
                    break
                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=(amp_dtype is not None)):
                    logits = app(token_ids, attention_mask=torch.ones_like(token_ids, device=DEVICE))[:, -1, :]
                if restrict_to_sequence_vocab:
                    logits = mask_to_sequence_vocab(logits, layout)
                next_token = sample_next_token(logits, temperature=temperature, top_k=top_k)
                next_id = int(next_token.item())
                generated_ids.append(next_id)
                token_ids = torch.cat([token_ids, next_token], dim=1)
                if verbose and progress_every > 0 and (step + 1) % int(progress_every) == 0:
                    print(f"generated {step + 1}/{max_new_tokens} tokens", flush=True)
                if next_id == layout.eos_token_id:
                    break

    sequence = decode_generated_sequence(artifacts, generated_ids)
    return {
        "sequence": sequence,
        "sequence_length": len(sequence),
        "prompt_tokens": prompt_length,
        "generated_tokens": len(generated_ids),
        "stopped_on_eos": bool(generated_ids and generated_ids[-1] == layout.eos_token_id),
        "used_cache": bool(use_cache),
    }


def generate_instruction_candidates(instruction: str, input_text: str = "", *, num_candidates: int = DEFAULT_NUM_CANDIDATES, **kwargs):
    results = []
    base_seed = kwargs.pop("seed", None)
    for index in range(int(num_candidates)):
        print(f"Generating instruction candidate {index + 1}/{num_candidates}...", flush=True)
        seed = None if base_seed is None else int(base_seed) + index
        item = generate_instruction_response(instruction, input_text, seed=seed, **kwargs)
        results.append(item)
        print(f"Done candidate {index + 1}: length={item['sequence_length']}, tokens={item['generated_tokens']}", flush=True)
    return results


In [16]:
instruction = (
    "Generate one soluble non-pathogenic E. coli-like cytosolic protein using ACDEFGHIKLMNPQRSTVWY only. "
    "Length 100-220 aa. Favor charged/polar residues. "
    "Avoid signal peptide, transmembrane, secretion, adhesion, toxin, virulence, resistance, immune-evasion, "
    "aerosol-survival, host-colonization, and low-complexity motifs. Output sequence only."
)

input_text = (
    "Computational screening only. Reject proteins associated with pathogenicity, transmission, "
    "airborne/aerosol fitness, secretion, membrane anchoring, toxicity, or antimicrobial resistance."
)
HYDROPHOBIC_AA = set("AILMFWVY")

MAX_LONGEST_HYDROPHOBIC_STRETCH = 10
MAX_TERMINAL_HYDROPHOBIC_FRACTION = 0.40

TARGET_ACCEPTED_CANDIDATES = 1
CANDIDATES_PER_ROUND = 4
MAX_GENERATION_ROUNDS = 8

REQUESTED_MAX_NEW_TOKENS = 256
MIN_GENERATION_TOKENS = 80


def longest_hydrophobic_stretch(sequence: str) -> int:
    longest = current = 0
    for aa in sequence:
        if aa in HYDROPHOBIC_AA:
            current += 1
            longest = max(longest, current)
        else:
            current = 0
    return longest


def quick_soluble_candidate_report(sequence: str) -> dict[str, object]:
    sequence = sequence.strip().upper()
    length = max(len(sequence), 1)
    n_term = sequence[:30]
    c_term = sequence[-30:]
    hydrophobic_fraction = sum(aa in HYDROPHOBIC_AA for aa in sequence) / length
    n_term_hydrophobic_fraction = sum(aa in HYDROPHOBIC_AA for aa in n_term) / max(len(n_term), 1)
    c_term_hydrophobic_fraction = sum(aa in HYDROPHOBIC_AA for aa in c_term) / max(len(c_term), 1)
    longest_stretch = longest_hydrophobic_stretch(sequence)
    reject_reasons = []
    if longest_stretch > MAX_LONGEST_HYDROPHOBIC_STRETCH:
        reject_reasons.append(f"longest hydrophobic stretch {longest_stretch} > {MAX_LONGEST_HYDROPHOBIC_STRETCH}")
    if n_term_hydrophobic_fraction >= MAX_TERMINAL_HYDROPHOBIC_FRACTION:
        reject_reasons.append(f"N-terminal hydrophobic fraction {n_term_hydrophobic_fraction:.3f} >= {MAX_TERMINAL_HYDROPHOBIC_FRACTION}")
    if c_term_hydrophobic_fraction >= MAX_TERMINAL_HYDROPHOBIC_FRACTION:
        reject_reasons.append(f"C-terminal hydrophobic fraction {c_term_hydrophobic_fraction:.3f} >= {MAX_TERMINAL_HYDROPHOBIC_FRACTION}")
    passes = not reject_reasons
    return {
        "hydrophobic_fraction": round(hydrophobic_fraction, 3),
        "n_term_hydrophobic_fraction": round(n_term_hydrophobic_fraction, 3),
        "c_term_hydrophobic_fraction": round(c_term_hydrophobic_fraction, 3),
        "longest_hydrophobic_stretch": longest_stretch,
        "passes_quick_soluble_filter": passes,
        "reject_reasons": reject_reasons,
    }


def instruction_prompt_token_count(instruction: str, input_text: str = "") -> int:
    return int(fused_instruction_prompt(RUNTIME["artifacts"], instruction, input_text).size(1))


def fit_instruction_prompt_to_context(
    instruction: str,
    input_text: str = "",
    *,
    min_generation_tokens: int = MIN_GENERATION_TOKENS,
) -> tuple[str, str]:
    context_tokens = int(RUNTIME["context_length"])
    target_prompt_tokens = context_tokens - int(min_generation_tokens)
    if target_prompt_tokens <= 0:
        raise ValueError(
            f"min_generation_tokens={min_generation_tokens} leaves no room for a prompt "
            f"in a {context_tokens}-token context window."
        )

    prompt_tokens = instruction_prompt_token_count(instruction, input_text)
    if prompt_tokens <= target_prompt_tokens:
        return instruction, input_text

    instruction_only_tokens = instruction_prompt_token_count(instruction, "")
    if instruction_only_tokens > target_prompt_tokens:
        raise ValueError(
            f"Instruction alone uses {instruction_only_tokens}/{context_tokens} context tokens. "
            f"Shorten the instruction or load a checkpoint with a larger context_length so at least "
            f"{min_generation_tokens} generation tokens remain."
        )

    normalized_input = " ".join(str(input_text or "").split())
    low = 0
    high = len(normalized_input)
    fitted_input_text = ""
    fitted_prompt_tokens = instruction_only_tokens
    while low <= high:
        mid = (low + high) // 2
        candidate = normalized_input[:mid].rstrip()
        candidate_tokens = instruction_prompt_token_count(instruction, candidate)
        if candidate_tokens <= target_prompt_tokens:
            fitted_input_text = candidate
            fitted_prompt_tokens = candidate_tokens
            low = mid + 1
        else:
            high = mid - 1

    if fitted_input_text and len(fitted_input_text) < len(normalized_input):
        word_boundary = fitted_input_text.rfind(" ")
        if word_boundary > 0:
            fitted_input_text = fitted_input_text[:word_boundary].rstrip()
            fitted_prompt_tokens = instruction_prompt_token_count(instruction, fitted_input_text)

    print(
        f"Trimmed input_text from {len(normalized_input)} to {len(fitted_input_text)} characters "
        f"to fit the {context_tokens}-token context window.",
        flush=True,
    )
    print(
        f"Fitted prompt uses {fitted_prompt_tokens}/{context_tokens} context tokens; "
        f"reserved generation tokens={context_tokens - fitted_prompt_tokens}.",
        flush=True,
    )
    return instruction, fitted_input_text


def context_safe_max_new_tokens(
    instruction: str,
    input_text: str = "",
    *,
    requested_max_new_tokens: int = REQUESTED_MAX_NEW_TOKENS,
    min_generation_tokens: int = MIN_GENERATION_TOKENS,
) -> int:
    prompt_tokens = instruction_prompt_token_count(instruction, input_text)
    context_tokens = int(RUNTIME["context_length"])
    available_tokens = context_tokens - prompt_tokens
    print(
        f"Prompt uses {prompt_tokens}/{context_tokens} context tokens; "
        f"available generation tokens={available_tokens}.",
        flush=True,
    )
    if available_tokens < int(min_generation_tokens):
        raise ValueError(
            f"Prompt leaves only {available_tokens} generation tokens. "
            f"Shorten instruction/input until at least {min_generation_tokens} are available."
        )
    safe_max_new_tokens = min(int(requested_max_new_tokens), available_tokens)
    print(f"Using max_new_tokens={safe_max_new_tokens} for this checkpoint.", flush=True)
    return safe_max_new_tokens


def generate_quick_soluble_candidates(
    instruction: str,
    input_text: str = "",
    *,
    target_accepted: int = TARGET_ACCEPTED_CANDIDATES,
    candidates_per_round: int = CANDIDATES_PER_ROUND,
    max_rounds: int = MAX_GENERATION_ROUNDS,
    seed: int | None = 123,
    **generation_kwargs,
) -> tuple[list[dict[str, object]], list[dict[str, object]]]:
    accepted = []
    rejected = []
    generated_count = 0

    for round_index in range(int(max_rounds)):
        if len(accepted) >= int(target_accepted):
            break

        round_seed = None if seed is None else int(seed) + generated_count
        print(f"\nGeneration round {round_index + 1}/{max_rounds}", flush=True)
        batch = generate_instruction_candidates(
            instruction,
            input_text,
            num_candidates=int(candidates_per_round),
            seed=round_seed,
            **generation_kwargs,
        )

        for item in batch:
            generated_count += 1
            report = quick_soluble_candidate_report(item["sequence"])
            item["quick_soluble_report"] = report
            if report["passes_quick_soluble_filter"]:
                accepted.append(item)
                print(f"ACCEPT candidate {generated_count}: length={item['sequence_length']}", flush=True)
            else:
                rejected.append(item)
                reasons = "; ".join(report["reject_reasons"])
                print(f"REJECT candidate {generated_count}: {reasons}", flush=True)

    print(
        f"\nAccepted {len(accepted)} passing candidate(s) after generating {generated_count}. "
        f"Target={target_accepted}.",
        flush=True,
    )
    return accepted, rejected


if RUNTIME["kind"] == "instruction":
    instruction, input_text = fit_instruction_prompt_to_context(
        instruction,
        input_text,
        min_generation_tokens=MIN_GENERATION_TOKENS,
    )
    max_new_tokens = context_safe_max_new_tokens(
        instruction,
        input_text,
        requested_max_new_tokens=REQUESTED_MAX_NEW_TOKENS,
        min_generation_tokens=MIN_GENERATION_TOKENS,
    )
    accepted_candidates, rejected_candidates = generate_quick_soluble_candidates(
        instruction,
        input_text,
        target_accepted=TARGET_ACCEPTED_CANDIDATES,
        candidates_per_round=CANDIDATES_PER_ROUND,
        max_rounds=MAX_GENERATION_ROUNDS,
        max_new_tokens=max_new_tokens,
        temperature=0.8,
        top_k=50,
        seed=123,
        verbose=True,
    )
    if accepted_candidates:
        print("\n=== RIGHT / PASSING PROTEIN SEQUENCE(S) ===")
        for i, item in enumerate(accepted_candidates, start=1):
            print(f"--- PASSING PROTEIN {i} | length={item['sequence_length']} ---")
            print(item["sequence"])
            print(json.dumps(item["quick_soluble_report"], indent=2))
        right_protein_sequence = accepted_candidates[0]["sequence"]
    else:
        print("\n=== NO PASSING PROTEIN FOUND; LAST REJECTED CANDIDATES ===")
        right_protein_sequence = None
        for i, item in enumerate(rejected_candidates[-min(3, len(rejected_candidates)):], start=1):
            print(f"--- REJECTED PROTEIN {i} | length={item['sequence_length']} ---")
            print(item["sequence"])
            print(json.dumps(item["quick_soluble_report"], indent=2))
else:
    print("No instruction checkpoint loaded. Train stage 3 first, or set MODEL_KIND='instruction'.")


Prompt uses 420/512 context tokens; available generation tokens=92.
Using max_new_tokens=92 for this checkpoint.

Generation round 1/8
Generating instruction candidate 1/4...
generated 16/92 tokens
generated 32/92 tokens
generated 48/92 tokens
generated 64/92 tokens
generated 80/92 tokens
Done candidate 1: length=125, tokens=92
Generating instruction candidate 2/4...
generated 16/92 tokens
generated 32/92 tokens
generated 48/92 tokens
generated 64/92 tokens
generated 80/92 tokens
Done candidate 2: length=160, tokens=92
Generating instruction candidate 3/4...
generated 16/92 tokens
generated 32/92 tokens
generated 48/92 tokens
Done candidate 3: length=95, tokens=61
Generating instruction candidate 4/4...
generated 16/92 tokens
generated 32/92 tokens
generated 48/92 tokens
generated 64/92 tokens
generated 80/92 tokens
Done candidate 4: length=131, tokens=92
ACCEPT candidate 1: length=125
REJECT candidate 2: N-terminal hydrophobic fraction 0.467 >= 0.4
REJECT candidate 3: N-terminal hydro

## Base Protein LM Inference

Use this now if you only have `protein_from_scratch` or `protein` checkpoints. The model continues a protein prompt such as `<|protein|>M`.

In [6]:
def generate_protein_continuation(
    prompt: str = f"{PROTEIN_START_TOKEN}M",
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    temperature: float = DEFAULT_TEMPERATURE,
    top_k: int | None = DEFAULT_TOP_K,
    seed: int | None = None,
    use_cache: bool = True,
    verbose: bool = False,
    progress_every: int = DEFAULT_PROGRESS_EVERY,
) -> dict[str, Any]:
    if RUNTIME["kind"] != "protein":
        raise RuntimeError("Current RUNTIME is not a base protein checkpoint. Set MODEL_KIND='protein'.")

    set_seed(seed)
    model = RUNTIME["model"]
    tokenizer = RUNTIME["tokenizer"]
    prompt_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=DEVICE).unsqueeze(0)
    eos_id = tokenizer.str_to_int[PROTEIN_END_TOKEN]
    max_context = int(RUNTIME["context_length"])
    amp_dtype = autocast_dtype()
    generated_ids: list[int] = []

    if prompt_ids.size(1) >= max_context:
        raise ValueError(f"Prompt length {prompt_ids.size(1)} is already >= context length {max_context}.")

    with torch.no_grad():
        if use_cache:
            cache = model.create_kv_cache()
            model.reset_kv_cache(cache)
            with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=(amp_dtype is not None)):
                logits = model(prompt_ids, cache=cache)[:, -1, :]

            for step in range(int(max_new_tokens)):
                if prompt_ids.size(1) + len(generated_ids) >= max_context:
                    break
                next_token = sample_next_token(logits, temperature=temperature, top_k=top_k)
                next_id = int(next_token.item())
                generated_ids.append(next_id)
                if verbose and progress_every > 0 and (step + 1) % int(progress_every) == 0:
                    print(f"generated {step + 1}/{max_new_tokens} tokens", flush=True)
                if next_id == eos_id:
                    break
                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=(amp_dtype is not None)):
                    logits = model(next_token, cache=cache)[:, -1, :]
        else:
            token_ids = prompt_ids
            for step in range(int(max_new_tokens)):
                if token_ids.size(1) >= max_context:
                    break
                current = token_ids[:, -max_context:]
                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=(amp_dtype is not None)):
                    logits = model(current, attn_mask=torch.ones_like(current, device=DEVICE))[:, -1, :]
                next_token = sample_next_token(logits, temperature=temperature, top_k=top_k)
                next_id = int(next_token.item())
                generated_ids.append(next_id)
                token_ids = torch.cat([token_ids, next_token], dim=1)
                if verbose and progress_every > 0 and (step + 1) % int(progress_every) == 0:
                    print(f"generated {step + 1}/{max_new_tokens} tokens", flush=True)
                if next_id == eos_id:
                    break

    all_ids = prompt_ids.squeeze(0).tolist() + generated_ids
    text = tokenizer.decode(all_ids)
    sequence = text.replace(PROTEIN_START_TOKEN, "").split(PROTEIN_END_TOKEN, 1)[0]
    return {
        "text": text,
        "sequence": sequence,
        "sequence_length": len(sequence),
        "tokens": len(all_ids),
        "generated_tokens": len(generated_ids),
        "stopped_on_eos": bool(generated_ids and generated_ids[-1] == eos_id),
        "used_cache": bool(use_cache),
    }


def generate_protein_candidates(prompt: str = f"{PROTEIN_START_TOKEN}M", *, num_candidates: int = DEFAULT_NUM_CANDIDATES, **kwargs):
    results = []
    base_seed = kwargs.pop("seed", None)
    for index in range(int(num_candidates)):
        print(f"Generating protein candidate {index + 1}/{num_candidates}...", flush=True)
        seed = None if base_seed is None else int(base_seed) + index
        item = generate_protein_continuation(prompt, seed=seed, **kwargs)
        results.append(item)
        print(f"Done candidate {index + 1}: length={item['sequence_length']}, tokens={item['generated_tokens']}", flush=True)
    return results


In [9]:
prompt = "<|protein|>MSSRKELANAIRALSMDAVQKAKSGHPGAPMGMADIAEVLWRDFL"

if RUNTIME["kind"] == "protein":
    candidates = generate_protein_candidates(
        prompt,
        num_candidates=1,
        max_new_tokens=1024,
        temperature=0.8,
        top_k=50,
        seed=123,
        verbose=True,
    )
    for i, item in enumerate(candidates, start=1):
        print(f"--- candidate {i} | length={item['sequence_length']} ---")
        print(item["sequence"])
else:
    print("Instruction checkpoint loaded. Use the instruction inference cell above.")


Generating protein candidate 1/1...
generated 16/1024 tokens
generated 32/1024 tokens
generated 48/1024 tokens
generated 64/1024 tokens
generated 80/1024 tokens
generated 96/1024 tokens
generated 112/1024 tokens
Done candidate 1: length=237, tokens=127
--- candidate 1 | length=237 ---
MSSRKELANAIRALSMDAVQKAKSGHPGAPMGMADIAEVLWRDFLKEKGYVRYRIDVDGGRYTAPLIADTADPVIAVTAATGCLVEDALRAEGVEYAVPDLPEIIGYMGIDHAFVQEFHTIEDAARALATEGRTTVIVPNHMQVHQAAGTRVALIDAVAPALQHAVTDLSRTERGWIEASHAARKDSLARAVDRLVARGYDAVVLHTADEGFTPESEEWGSFPDEVADALEELVDLG


## Cleanup

In [ ]:
# Uncomment and run manually when switching checkpoints in the same kernel.
# del RUNTIME
# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
